## tl;dr

개방ID `5622779190`은 2009~2025년 서울 종로구 대학원으로 남아 있다. 2025년 10개 패널·363행 중 실질 비영 값은 `행정학과`의 `지정형전임교원수=1` 한 건뿐이며, 13개 학과는 모두 `폐과`다. 학과명과 소재지는 2004년에 폐지된 성균관대학교 행정대학원과 강하게 일치한다.

## Context & Methods

EDSS DuckDB에서 2025년 패널별 행 수와 비영 측정값, 0101 연도별 핵심 지표, 1017 학과 상태를 확인하고 기존 KEDI 매핑 파일과 대조한다. 학교 후보는 성균관대학교 공식 연혁으로 교차검증한다.

### Key Assumptions

- 개방ID는 문자열로 처리한다.
- 명시적 0과 결측치는 구분한다.
- 학교 후보는 자동 조인이 아니라 수동 검토 결론이다.
- 공식 연혁: https://gsg.skku.edu/gsg/graduate/gov_history.do

In [1]:
import csv
import decimal
import os
from pathlib import Path
import duckdb

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
database_path = Path(os.environ.get(
    'EDSS_DUCKDB_PATH',
    '/Users/joocheol/Documents/GitHub/edss/data/processed/edss/restricted/edss_all.duckdb',
))
identity_path = repo_root / 'data/processed/edss_0101_kedi_openid_identity_2009_2025.csv'
evidence_path = repo_root / 'data/processed/edss_0101_kedi_row_match_evidence_2009_2025.csv'
assert database_path.exists() and identity_path.exists() and evidence_path.exists()
connection = duckdb.connect(str(database_path), read_only=True)
open_id = '5622779190'
duckdb.__version__, database_path.name

('1.4.1', 'edss_all.duckdb')

## Results

### 1. 2025년 패널별 행 수

In [2]:
tables = connection.execute("""
SELECT table_schema, table_name
FROM information_schema.columns
WHERE column_name = '개방ID'
  AND table_schema IN ('higher_education', 'university_disclosure')
ORDER BY table_schema, table_name
""").fetchall()
panel_rows = []
for schema_name, table_name in tables:
    row_count = connection.execute(
        f"SELECT COUNT(*) FROM {schema_name}.{table_name} WHERE 개방ID=? AND 조사년도='2025'",
        [open_id],
    ).fetchone()[0]
    if row_count:
        panel_rows.append((schema_name, table_name, row_count))
assert len(panel_rows) == 10
assert sum(row[2] for row in panel_rows) == 363
panel_rows

[('higher_education', 'panel_0101', 1),
 ('higher_education', 'panel_0104', 1),
 ('higher_education', 'panel_0105', 2),
 ('higher_education', 'panel_0231', 26),
 ('higher_education', 'panel_0246', 12),
 ('university_disclosure', 'panel_0306', 3),
 ('university_disclosure', 'panel_0308', 5),
 ('university_disclosure', 'panel_0715', 234),
 ('university_disclosure', 'panel_1010', 1),
 ('university_disclosure', 'panel_1017', 78)]

### 2. 2025년 비영 측정값

In [3]:
dimension_columns = {
    '조사년도', '개방ID', '적용년도', '학기구분명', '수업연한명', '개설기간명',
    '학년명', '성별명', '학위과정구분명', '학과명', '학과한글명', '학과상태명',
    '단과대학명', '교육부계열명', '학과계열구분명', '주야간계절구분명',
    '본분교명', '시도명', '지역명', '학교구분명', '학제유형명', '교원구분명',
}
nonzero_measure_cells = []
for schema_name, table_name, _ in panel_rows:
    cursor = connection.execute(
        f"SELECT * FROM {schema_name}.{table_name} WHERE 개방ID=? AND 조사년도='2025'",
        [open_id],
    )
    rows = cursor.fetchall()
    columns = [item[0] for item in cursor.description]
    for column_index, column_name in enumerate(columns):
        if column_name.startswith('_') or column_name in dimension_columns:
            continue
        for row in rows:
            value = row[column_index]
            try:
                numeric_value = decimal.Decimal(str(value).replace(',', ''))
            except decimal.InvalidOperation:
                continue
            if numeric_value != 0:
                nonzero_measure_cells.append((schema_name, table_name, column_name, value))
assert nonzero_measure_cells == [('university_disclosure', 'panel_1010', '지정형전임교원수', '1')]
nonzero_measure_cells

[('university_disclosure', 'panel_1010', '지정형전임교원수', '1')]

In [4]:
teacher_row = connection.execute("""
SELECT 학과한글명, 단과대학명, 지정형전임교원수, 채용형전임교원수
FROM university_disclosure.panel_1010
WHERE 개방ID=? AND 조사년도='2025'
""", [open_id]).fetchall()
assert teacher_row == [('행정학과', '원천코드없음', '1', '0')]
teacher_row

[('행정학과', '원천코드없음', '1', '0')]

### 3. 0101 연도별 핵심 지표

In [5]:
yearly_rows = connection.execute("""
SELECT 조사년도, 지역명, 고등교육학교_재적학생수, 고등교육학교_학과수,
       고등교육학교_교원수, 고등교육학교_졸업생수
FROM higher_education.panel_0101
WHERE 개방ID=?
ORDER BY 조사년도
""", [open_id]).fetchall()
assert len(yearly_rows) == 17
assert {row[1] for row in yearly_rows} == {'서울 종로구'}
exceptions = [row for row in yearly_rows if any(value != '0' for value in row[2:])]
exceptions

[('2021', '서울 종로구', '0', '0', '0', '1'),
 ('2024', '서울 종로구', '0', '1', '0', '0')]

### 4. 폐과 학과 목록

In [6]:
department_rows = connection.execute("""
SELECT DISTINCT 학과명, 학과상태명
FROM university_disclosure.panel_1017
WHERE 개방ID=? AND 조사년도='2025'
ORDER BY 학과명
""", [open_id]).fetchall()
assert len(department_rows) == 13
assert {status for _, status in department_rows} == {'폐과'}
department_rows

[('감사행정학과', '폐과'),
 ('공공감사학과', '폐과'),
 ('공공정책전공', '폐과'),
 ('공안행정학과', '폐과'),
 ('교통물류학과', '폐과'),
 ('교통행정학과', '폐과'),
 ('부동산행정학과', '폐과'),
 ('의료행정학과', '폐과'),
 ('정보시스템감사전공', '폐과'),
 ('정보정책학과', '폐과'),
 ('정책학과', '폐과'),
 ('행정관리전공', '폐과'),
 ('행정학과', '폐과')]

### 5. 기존 KEDI 매핑 상태

In [7]:
with identity_path.open(encoding='utf-8-sig', newline='') as handle:
    identity_rows = [row for row in csv.DictReader(handle) if row['openid'] == open_id]
with evidence_path.open(encoding='utf-8-sig', newline='') as handle:
    evidence_rows = [row for row in csv.DictReader(handle) if row['openid'] == open_id]
assert len(identity_rows) == 1
assert identity_rows[0]['identity_status'] == 'unmatched'
assert evidence_rows == []
identity_rows[0], len(evidence_rows)

({'openid': '5622779190',
  'first_edss_year': '2009',
  'last_edss_year': '2025',
  'edss_year_count': '17',
  'direct_match_year_count': '0',
  'latest_direct_school_name': '',
  'kedi_school_code_2025': '',
  'distinct_normalized_name_count': '0',
  'name_history': '',
  'identity_status': 'unmatched'},
 0)

## Takeaways

- 2025년 10개 패널·363행 중 비영 측정값은 행정학과 지정형 전임교원 1명뿐이다.
- 2025년 13개 학과가 모두 폐과이며, 0101의 재학생·교원은 전 연도 0이다.
- 서울 종로구, 감사행정학과→공공감사학과, 교통물류·부동산행정·의료행정 등의 조합은 성균관대학교 옛 행정대학원과 일치한다.
- 성균관대학교 공식 연혁은 행정대학원이 2004년 폐지되고 국정관리대학원이 신설되었다고 명시한다. 따라서 `성균관대학교 행정대학원 폐교·폐과 잔존 ID`가 가장 유력하지만, 기존 KEDI 직접 매핑 근거는 없으므로 자동 조인은 하지 않는다.